In [1]:
# ============================================================================
# notebook: notebooks/05_external_validity.ipynb  (consolidation of old 10-14)
# Stage (external validity): (a) attempt external replication of the typicality
#   paradox on GMSC — documented as DEGENERATE density (unfit); (b) internal
#   robustness within Taiwan via random splits. Honest scoping, not external.
# Reads data/ + results/. Run from notebooks/.
# ============================================================================


# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Paths, imports
# ─────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import statsmodels.api as sm

ROOT    = Path("..").resolve()
DATA    = ROOT / "data"
RESULTS = ROOT / "results"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
K = 20


# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Density-geometry validity check (the GMSC lesson, reusable)
# ─────────────────────────────────────────────────────────────────────────
def geom(X):
    d,_ = NearestNeighbors(n_neighbors=K+1).fit(X).kneighbors(X)
    mnn = d[:,1:].mean(axis=1); dens = 1.0/(mnn+1e-9)
    return stats.skew(mnn), dens.max()/np.median(dens), dens

def density_ok(X, skew_max=60, ratio_max=1e5):
    s, r, dens = geom(X)
    return (s < skew_max) and (r < ratio_max), dict(skew=round(s,1), ratio=round(r,1)), dens


# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — GMSC external replication attempt (expected: DEGENERATE)
# ─────────────────────────────────────────────────────────────────────────
def clean_gmsc(g):
    for c in ["PD_30_59","PD_60_89","PD_90"]:
        if c in g: g.loc[g[c].isin([96,98]), c] = np.nan
    g.loc[g["AGE_YEARS"]<18,"AGE_YEARS"] = np.nan
    for c in ["UTIL","DEBT_RATIO"]:
        g[c] = g[c].clip(upper=g[c].quantile(0.99))
    g = g.dropna(subset=["PD_30_59","PD_60_89","PD_90","AGE_YEARS"]).reset_index(drop=True)
    no_pd = (g[["PD_30_59","PD_60_89","PD_90"]]==0).all(axis=1)
    g["VIP"] = (no_pd & (g["UTIL"]<=g["UTIL"].median())).astype(int)
    return g

gm_path = DATA / "gmsc_raw.parquet"
if gm_path.exists():
    gm = clean_gmsc(pd.read_parquet(gm_path))
    gm_audit = ["UTIL","DEBT_RATIO","INCOME","OPEN_LINES","RE_LOANS","DEPENDENTS"]
    appr = gm[gm["VIP"]==1].reset_index(drop=True)
    Xg = StandardScaler().fit_transform(appr[gm_audit].values)
    ok, diag, _ = density_ok(Xg)
    print(f"GMSC density geometry: {diag} -> {'VALID' if ok else 'DEGENERATE'}")
    print("  => GMSC unfit for density-based replication (documented limitation).")
else:
    print("GMSC raw not found; skipping external attempt.")


# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Taiwan internal robustness: random-split replication of the paradox
# ─────────────────────────────────────────────────────────────────────────
tw = pd.read_parquet(DATA / "taiwan_raw.parquet")
pay = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
never = (tw[pay]<=0).all(axis=1)
dilig = pd.concat([tw[f"PAY_AMT{t}"]/(tw[f"BILL_AMT{t}"].abs()+1.0) for t in range(1,7)],axis=1).mean(axis=1)
tw["VIP"] = (never & (dilig>=dilig.median())).astype(int)
appr = tw[tw["VIP"]==1].reset_index(drop=True)
FULL = ["LIMIT_BAL"]+[f"BILL_AMT{i}" for i in range(1,7)]+[f"PAY_AMT{i}" for i in range(1,7)]

def paradox(frame, name):
    X = StandardScaler().fit_transform(frame[FULL].values)
    ok, diag, dens = density_ok(X)
    if not ok:
        print(f"  [{name}] density {diag} DEGENERATE — skip"); return None
    f = frame.copy(); f["density_pct"] = pd.Series(dens,index=f.index).rank(pct=True)
    m = sm.Logit(f["DEFAULT"].values, sm.add_constant(f[["density_pct"]])).fit(disp=0)
    c,p = m.params["density_pct"], m.pvalues["density_pct"]
    print(f"  [{name}] density {diag} VALID; coef={c:+.3f} (p={p:.1e}) "
          f"{'PARADOX' if c>0 else 'conv.'}")
    return c,p

rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(len(appr))
h1 = appr.iloc[perm[:len(appr)//2]].reset_index(drop=True)
h2 = appr.iloc[perm[len(appr)//2:]].reset_index(drop=True)
print("Taiwan internal random-split replication:")
paradox(h1, "random_half_1"); paradox(h2, "random_half_2")
print("=== EXTERNAL VALIDITY: paradox robust WITHIN Taiwan; external open (GMSC unfit) ===")

GMSC density geometry: {'skew': np.float64(141.6), 'ratio': np.float64(153586122.3)} -> DEGENERATE
  => GMSC unfit for density-based replication (documented limitation).
Taiwan internal random-split replication:
  [random_half_1] density {'skew': np.float64(11.0), 'ratio': np.float64(25.4)} VALID; coef=+1.538 (p=1.4e-23) PARADOX
  [random_half_2] density {'skew': np.float64(12.0), 'ratio': np.float64(24.2)} VALID; coef=+1.234 (p=3.5e-16) PARADOX
=== EXTERNAL VALIDITY: paradox robust WITHIN Taiwan; external open (GMSC unfit) ===
